# Projectile with air drag

*mechanics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/mechanics/projectile)

Newton's second law for a ball with quadratic air drag. Several launch angles are solved at once by stacking them into one batched tensor, so a single einsum handles every trajectory.

**Vector form**

$$m\,\ddot{\mathbf r} = m\,\mathbf g - c\,\lVert\dot{\mathbf r}\rVert\,\dot{\mathbf r}$$

**Index (tensor) form**

$$\ddot x_{bi} = g_i - \tfrac{c}{m}\,\sqrt{v_{bj}v_{bj}}\;v_{bi}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $x_{bi}$ | `y[:, 0, :]` | (B, 2) | position of ball $b$, component $i \in \{x, z\}$ |
| $v_{bi}$ | `y[:, 1, :]` | (B, 2) | velocity of ball $b$ |
| $g_i$ | `g` | (2,) | gravity vector $(0, -9.81)$ |
| $v_{bj}v_{bj}$ | `einsum('bj,bj->b', v, v)` | (B,) | repeated $j$ ⇒ summed: squared speed of each ball |
| $\ddot x_{bi}$ | `acceleration(v, g, k)` | (B, 2) | right-hand side of the ODE |

The batch index $b$ is not physics: it is the tensor trick that lets PyTorch solve $B$ projectiles in one pass. Setting the drag to zero recovers the textbook parabola with maximum range at 45°.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/mechanics/projectile/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    g = tensors.gravity(dev)
    angles = tensors.launch_angles(params["n_angles"])
    y0 = tensors.initial_state(params["v0"], angles, dev)

    times, traj = solve.solve(y0, g, params["drag"], params["t_max"])

    for deg, r in zip(angles.rad2deg().tolist(), solve.ranges(traj).tolist()):
        print(f"angle {deg:5.1f}°  range {r:7.2f} m")
    return plot.animate(times, traj, angles, out_dir / "projectile.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/mechanics/projectile/equation.py
r"""Physics → tensor translation.

Newton's second law with quadratic drag,

    m \ddot{\mathbf r} = m \mathbf g - c \lVert \dot{\mathbf r} \rVert \dot{\mathbf r},

written in index notation for a batch of balls b and spatial components i:

    \ddot x_{bi} = g_i - (c/m) \sqrt{v_{bj} v_{bj}} \, v_{bi}

A repeated index (j above) is summed — that is exactly what einsum('bj,bj->b') does.
As a first-order system with state y = (x, v):

    \dot x_{bi} = v_{bi},    \dot v_{bi} = \ddot x_{bi}
"""

import torch


def acceleration(v: torch.Tensor, g: torch.Tensor, k: float) -> torch.Tensor:
    r"""\ddot x_{bi} = g_i - k \sqrt{v_{bj} v_{bj}} v_{bi}   with k = c/m.   v: (B, 2) → (B, 2)"""
    speed = torch.sqrt(torch.einsum("bj,bj->b", v, v))  # (B,)   |v_b|
    return g - k * torch.einsum("b,bi->bi", speed, v)  # (B, 2) broadcast g_i over b


def rhs(y: torch.Tensor, g: torch.Tensor, k: float) -> torch.Tensor:
    """dy/dt for the state y of shape (B, 2, 2): y[:, 0] = position, y[:, 1] = velocity."""
    x_dot = y[:, 1]
    v_dot = acceleration(y[:, 1], g, k)
    return torch.stack([x_dot, v_dot], dim=1)


def energy(y: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
    r"""Mechanical energy per unit mass  E_b = ½ v_{bi} v_{bi} - g_i x_{bi}  (conserved when k = 0)."""
    x, v = y[..., 0, :], y[..., 1, :]
    return 0.5 * torch.einsum("...i,...i->...", v, v) - torch.einsum("i,...i->...", g, x)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/mechanics/projectile/tensors.py
"""Build the concrete tensors: gravity vector and the batched initial state."""

import math

import torch


def gravity(device: torch.device) -> torch.Tensor:
    """g_i with i ∈ {x, z}: shape (2,)."""
    return torch.tensor([0.0, -9.81], device=device)


def launch_angles(n: int) -> torch.Tensor:
    """n angles spread between 15° and 75° (or just 45° for a single ball), in radians."""
    if n == 1:
        return torch.tensor([math.radians(45.0)])
    return torch.deg2rad(torch.linspace(15.0, 75.0, n))


def initial_state(v0: float, angles: torch.Tensor, device: torch.device) -> torch.Tensor:
    """y0 of shape (B, 2, 2): every ball starts at the origin with speed v0 at its own angle.

    y0[b, 0, :] = (0, 0)                            position x_{bi}
    y0[b, 1, :] = v0 (cos θ_b, sin θ_b)             velocity v_{bi}
    """
    angles = angles.to(device)
    velocity = v0 * torch.stack([torch.cos(angles), torch.sin(angles)], dim=-1)  # (B, 2)
    position = torch.zeros_like(velocity)
    return torch.stack([position, velocity], dim=1)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/mechanics/projectile/solve.py
"""Integrate the batched ODE with 4th-order Runge–Kutta and stop each ball at the ground."""

import torch

from problems._common.integrators import integrate

from .equation import rhs


def solve(
    y0: torch.Tensor, g: torch.Tensor, k: float, t_max: float, dt: float = 0.01
) -> tuple[torch.Tensor, torch.Tensor]:
    """Return (times (T,), trajectory (T, B, 2, 2)).

    All B balls advance together: one RK4 step updates the whole (B, 2, 2) tensor.
    After a ball hits the ground (z < 0) we freeze it at its landing point.
    """
    n_steps = int(round(t_max / dt))
    traj = integrate(lambda y, t: rhs(y, g, k), y0, dt, n_steps)  # (T, B, 2, 2)

    z = traj[:, :, 0, 1]  # (T, B) height of each ball
    landed = torch.cummax((z < 0).int(), dim=0).values.bool()  # stays True after first touchdown
    landed[0] = False
    step_of_landing = torch.where(landed.any(0), landed.int().argmax(0), len(z) - 1)  # (B,)
    landing = traj[step_of_landing, torch.arange(traj.shape[1])].clone()  # (B, 2, 2)
    landing[:, 0, 1] = 0.0  # on the ground
    landing[:, 1] = 0.0  # at rest
    traj = torch.where(landed[:, :, None, None], landing, traj)

    times = torch.arange(n_steps + 1, device=y0.device) * dt
    return times, traj


def ranges(traj: torch.Tensor) -> torch.Tensor:
    """Horizontal distance travelled by each ball at the end of the run: (B,)."""
    return traj[-1, :, 0, 0]


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/mechanics/projectile/plot.py
"""Animate the trajectories: each ball traces its path, labelled by launch angle."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import frame_indices, save_gif


def animate(times: torch.Tensor, traj: torch.Tensor, angles: torch.Tensor, path: Path) -> Path:
    pos = traj[:, :, 0, :].cpu().numpy()  # (T, B, 2)
    t = times.cpu().numpy()
    degrees = torch.rad2deg(angles).cpu().numpy()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.set_xlim(0, max(1.0, pos[..., 0].max() * 1.05))
    ax.set_ylim(0, max(1.0, pos[..., 1].max() * 1.15))
    ax.set_xlabel("x (m)")
    ax.set_ylabel("z (m)")
    ax.set_aspect("equal", adjustable="box")

    lines = [ax.plot([], [], lw=1.8, label=f"{d:.0f}°")[0] for d in degrees]
    dots = [ax.plot([], [], "o", color=line.get_color())[0] for line in lines]
    clock = ax.text(0.02, 0.95, "", transform=ax.transAxes, va="top")
    ax.legend(loc="upper right", fontsize=8, title="launch angle")

    def update(step: int):
        for b, (line, dot) in enumerate(zip(lines, dots)):
            line.set_data(pos[: step + 1, b, 0], pos[: step + 1, b, 1])
            dot.set_data([pos[step, b, 0]], [pos[step, b, 1]])
        clock.set_text(f"t = {t[step]:.2f} s")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(t)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'v0': 30.0,
    'drag': 0.02,
    'n_angles': 6,
    't_max': 6.0
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.mechanics.projectile")]:
    del sys.modules[name]
main = importlib.import_module("problems.mechanics.projectile.main")
problem_dir = Path("problems/mechanics/projectile")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
